# 01. Загрузка данных из разных форматов

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В этом ноутбуке мы научимся загружать данные из разных файловых форматов:

- CSV;
- Excel;
- JSON;
- HTML-таблица.

Также разберем важные параметры чтения:

- `sep`;
- `encoding`;
- `sheet_name`;
- `header`;
- `index_col`.

В конце сохраним данные обратно в CSV и Excel.

## 1. Цель ноутбука

После выполнения этого ноутбука вы должны уметь:

1. Загружать CSV-файл через `pd.read_csv()`.
2. Понимать, зачем нужны параметры `sep` и `encoding`.
3. Загружать Excel-файл через `pd.read_excel()`.
4. Выбирать нужный лист Excel через `sheet_name`.
5. Загружать JSON-файл через `pd.read_json()`.
6. Загружать HTML-таблицу через `pd.read_html()`.
7. Понимать параметры `header` и `index_col`.
8. Сохранять результат в CSV и Excel.

Главная идея:

> Файл на диске превращается в таблицу pandas — `DataFrame`.

## 2. Какие файлы используются

В этом ноутбуке используются учебные данные кейса **«РегионМаркет»**.

Ожидаемые файлы:

```text
data/raw/sales.csv
data/raw/products.xlsx
data/raw/regions.json
data/raw/clients.csv
data/raw/web_table_sample.html
```

Если вы работаете с архивом датасетов напрямую, файлы могут лежать в папке:

```text
data/raw/
```

В коде ниже мы попробуем найти данные автоматически.

## 3. Импорт библиотек

Для загрузки данных нам нужна библиотека `pandas`.

Также используем `Path` из стандартной библиотеки Python, чтобы удобно работать с путями к файлам.

In [ ]:
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

## 4. Поиск папки с данными

В реальных проектах важно понимать, где лежат файлы.

Сейчас мы найдем папку с исходными данными. Обычно это:

```text
data/raw/
```

Если такой папки нет, попробуем найти папку:

```text
data/raw/
```

In [ ]:
def find_data_dir() -> Path:
    """Найти папку с учебными данными."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if (candidate / "sales.csv").exists():
            return candidate

    return current_dir / "data" / "raw"


DATA_DIR = find_data_dir()

print("Папка с данными:")
print(DATA_DIR)

print("\nФайлы в папке:")
if DATA_DIR.exists():
    for item in sorted(DATA_DIR.iterdir()):
        print("-", item.name)
else:
    print("Папка не найдена. Проверьте структуру проекта.")

## 5. Что такое CSV

**CSV** — это текстовый формат для хранения таблиц.

Обычно одна строка файла соответствует одной строке таблицы, а значения внутри строки разделяются специальным символом.

Частые разделители:

| Разделитель | Где встречается |
|---|---|
| `,` | часто в англоязычных CSV |
| `;` | часто в русскоязычных Excel-выгрузках |
| `\t` | TSV-файлы, где значения разделены табуляцией |

Файл `sales.csv` содержит факты продаж.

## 6. Загрузка CSV через `pd.read_csv()`

Прочитаем файл продаж.

In [ ]:
sales_path = DATA_DIR / "sales.csv"

sales = pd.read_csv(sales_path)

print("Файл прочитан:", sales_path.name)
print("Размер таблицы:", sales.shape)

sales.head()

### Что произошло

Команда:

```python
pd.read_csv(sales_path)
```

прочитала CSV-файл и вернула объект `DataFrame`.

`DataFrame` — это таблица в pandas:

- строки — наблюдения, события или записи;
- столбцы — признаки, поля или переменные.

## 7. Быстрая проверка загруженной таблицы

После загрузки файла нужно посмотреть:

- первые строки;
- размер таблицы;
- список столбцов;
- типы данных.

In [ ]:
print("Размер:", sales.shape)

print("\nСтолбцы:")
print(sales.columns.tolist())

print("\nИнформация о таблице:")
sales.info()

## 8. Параметр `sep`

`sep` указывает, каким символом разделены значения в CSV-файле.

По умолчанию pandas ожидает запятую:

```python
pd.read_csv("file.csv")
```

Это примерно то же самое, что:

```python
pd.read_csv("file.csv", sep=",")
```

Если файл разделен точкой с запятой, нужно указать:

```python
pd.read_csv("file.csv", sep=";")
```

In [ ]:
# В нашем sales.csv используется запятая, поэтому эти два варианта должны дать одинаковый результат.

sales_default = pd.read_csv(sales_path)
sales_with_sep = pd.read_csv(sales_path, sep=",")

print("Размер без явного sep:", sales_default.shape)
print("Размер с sep=',':", sales_with_sep.shape)

### Типичная ошибка с `sep`

Если указать неправильный разделитель, таблица может загрузиться как один большой столбец.

Например, если файл разделен запятыми, а мы укажем `sep=";"`, pandas не сможет правильно разбить строки на столбцы.

In [ ]:
sales_wrong_sep = pd.read_csv(sales_path, sep=";")

print("Размер при неправильном sep:", sales_wrong_sep.shape)
sales_wrong_sep.head()

### Что нужно заметить

Если после чтения CSV получился **один столбец вместо многих**, вероятная причина — неверный разделитель.

Проверка:

```python
df.shape
df.head()
```

## 9. Параметр `encoding`

`encoding` — это кодировка файла.

Кодировка определяет, как байты файла превращаются в символы.

Частые варианты:

| Кодировка | Где встречается |
|---|---|
| `utf-8` | современный стандарт |
| `utf-8-sig` | CSV из Excel с BOM |
| `cp1251` | старые русскоязычные Windows-выгрузки |

Если кодировка указана неверно, может появиться ошибка `UnicodeDecodeError` или некорректные символы.

In [ ]:
# Обычно наш файл читается как utf-8.
# Явно укажем encoding, чтобы показать синтаксис.

sales_utf8 = pd.read_csv(sales_path, encoding="utf-8")

print("Файл прочитан с encoding='utf-8'")
sales_utf8.head()

### Что делать при ошибке кодировки

Если появилась ошибка:

```text
UnicodeDecodeError
```

можно попробовать:

```python
pd.read_csv("file.csv", encoding="utf-8")
pd.read_csv("file.csv", encoding="utf-8-sig")
pd.read_csv("file.csv", encoding="cp1251")
```

На занятии важно понять не все кодировки, а сам принцип:

> если файл не читается из-за символов, возможно, проблема в кодировке.

## 10. Параметр `header`

`header` указывает, какая строка файла содержит названия столбцов.

По умолчанию:

```python
header=0
```

Это значит: первая строка файла — заголовок таблицы.

Иногда файл может быть без заголовка. Тогда используют:

```python
header=None
```

In [ ]:
# Обычное чтение: первая строка считается заголовком.
sales_header_0 = pd.read_csv(sales_path, header=0)

print("Столбцы при header=0:")
print(sales_header_0.columns.tolist())

In [ ]:
# Демонстрация: прочитаем файл так, как будто заголовка нет.
# Это не правильный способ для нашего файла, а учебный пример.

sales_no_header = pd.read_csv(sales_path, header=None)

print("Столбцы при header=None:")
print(sales_no_header.columns.tolist())

sales_no_header.head()

### Что нужно заметить

Если указать `header=None`, pandas не использует первую строку как названия столбцов.

В результате:

- столбцы получают номера `0`, `1`, `2`, ...
- первая строка с названиями столбцов становится обычной строкой данных.

Это полезно для файлов без заголовков, но вредно для файлов, где заголовок уже есть.

## 11. Параметр `index_col`

`index_col` указывает, какой столбец использовать как индекс таблицы.

Индекс — это метки строк в DataFrame.

Для начинающих важно помнить:

> чаще всего на первых занятиях лучше не назначать индекс вручную, а оставить обычную нумерацию строк.

Но иногда удобно сделать идентификатор индексом.

In [ ]:
sales_indexed = pd.read_csv(sales_path, index_col="sale_id")

print("Индекс таблицы:")
print(sales_indexed.index[:5])

sales_indexed.head()

### Когда использовать `index_col`

Можно использовать, если:

- в данных есть стабильный уникальный идентификатор строки;
- вы точно понимаете, зачем он нужен как индекс.

Не стоит использовать, если:

- в столбце есть дубликаты;
- идентификатор понадобится для `merge`;
- вы только начинаете анализ.

В нашем кейсе `sale_id` содержит учебный дубликат, поэтому назначение его индексом может быть не лучшим решением.

## 12. Загрузка Excel-файла

Файл `products.xlsx` — справочник товаров.

Excel-файл может содержать несколько листов. Поэтому при чтении часто используют параметр `sheet_name`.

In [ ]:
products_path = DATA_DIR / "products.xlsx"

# Посмотрим список листов Excel-файла.
excel_file = pd.ExcelFile(products_path)

print("Листы Excel-файла:")
print(excel_file.sheet_names)

## 13. Параметр `sheet_name`

`sheet_name` указывает, какой лист Excel-файла нужно прочитать.

Варианты:

```python
pd.read_excel("file.xlsx", sheet_name="products")
pd.read_excel("file.xlsx", sheet_name=0)
```

- `sheet_name="products"` — прочитать лист по имени;
- `sheet_name=0` — прочитать первый лист.

In [ ]:
products = pd.read_excel(products_path, sheet_name="products")

print("Файл прочитан:", products_path.name)
print("Размер таблицы:", products.shape)

products.head()

## 14. Чтение другого листа Excel

В нашем Excel-файле есть дополнительный лист `category_reference`.

Он нужен как пример справочника категорий.

In [ ]:
category_reference = pd.read_excel(products_path, sheet_name="category_reference")

print("Размер таблицы category_reference:", category_reference.shape)

category_reference.head()

## 15. Загрузка JSON-файла

JSON часто используется в веб-сервисах, API и системных интеграциях.

Файл `regions.json` содержит справочник регионов.

In [ ]:
regions_path = DATA_DIR / "regions.json"

regions = pd.read_json(regions_path)

print("Файл прочитан:", regions_path.name)
print("Размер таблицы:", regions.shape)

regions.head()

### Что важно понять про JSON

JSON может быть:

- простым списком объектов;
- вложенной структурой;
- ответом API;
- конфигурационным файлом.

В нашем учебном примере JSON простой: список регионов, где каждый объект превращается в строку таблицы.

## 16. Загрузка еще одного CSV: справочник клиентов

Файл `clients.csv` — это справочник клиентов из CRM.

Загрузим его отдельно.

In [ ]:
clients_path = DATA_DIR / "clients.csv"

clients = pd.read_csv(clients_path, encoding="utf-8")

print("Файл прочитан:", clients_path.name)
print("Размер таблицы:", clients.shape)

clients.head()

## 17. Загрузка HTML-таблицы

HTML-таблицы можно читать через `pd.read_html()`.

В отличие от `read_csv()` или `read_excel()`, функция `read_html()` возвращает **список таблиц**, потому что на одной HTML-странице может быть несколько таблиц.

In [ ]:
html_path = DATA_DIR / "web_table_sample.html"

html_tables = pd.read_html(html_path)

print("Количество таблиц на HTML-странице:", len(html_tables))

## 18. Выбор нужной HTML-таблицы

Если на странице одна таблица, она будет первой в списке:

```python
html_tables[0]
```

In [ ]:
plans = html_tables[0]

print("Размер таблицы планов:", plans.shape)

plans.head()

## 19. Проверяем, что все данные загружены

Теперь у нас есть несколько таблиц:

- `sales`;
- `products`;
- `regions`;
- `clients`;
- `plans`.

Пока мы только загрузили данные. Объединять их будем в следующем ноутбуке.

In [ ]:
datasets = {
    "sales": sales,
    "products": products,
    "regions": regions,
    "clients": clients,
    "plans": plans,
}

for name, df in datasets.items():
    print(f"{name:<10} строк: {df.shape[0]:>3}, столбцов: {df.shape[1]:>2}")

## 20. Универсальная функция первичного осмотра

Чтобы быстро смотреть любую таблицу, создадим простую функцию.

In [ ]:
def quick_look(df: pd.DataFrame, name: str) -> None:
    """Показать базовую информацию о таблице."""
    print("=" * 70)
    print(f"Таблица: {name}")
    print("=" * 70)
    print("Размер:", df.shape)
    print("\nСтолбцы:")
    print(df.columns.tolist())
    print("\nТипы данных:")
    print(df.dtypes)
    print("\nПропуски:")
    print(df.isna().sum())
    print()

In [ ]:
quick_look(sales, "sales")

### Зачем нужна такая функция

В реальной работе аналитик часто повторяет одни и те же действия:

```python
df.shape
df.head()
df.info()
df.isna().sum()
```

Функция помогает не копировать один и тот же код много раз.

## 21. Сохранение данных в CSV

После загрузки и первичной проверки данные часто нужно сохранить.

Например, сохраним первые 10 строк продаж в отдельный файл.

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

sales_sample = sales.head(10)

csv_output_path = output_dir / "sales_sample.csv"

sales_sample.to_csv(csv_output_path, index=False, encoding="utf-8")

print("Файл сохранен:")
print(csv_output_path)

### Почему `index=False`

Если не указать `index=False`, pandas сохранит индекс DataFrame как отдельный столбец.

Обычно при сохранении обычной таблицы это не нужно.

Поэтому часто пишут:

```python
df.to_csv("file.csv", index=False)
```

## 22. Сохранение данных в Excel

Теперь сохраним тот же фрагмент в Excel-файл.

In [ ]:
excel_output_path = output_dir / "sales_sample.xlsx"

sales_sample.to_excel(excel_output_path, index=False, sheet_name="sales_sample")

print("Файл сохранен:")
print(excel_output_path)

## 23. Сохранение нескольких таблиц в один Excel-файл

Excel-файл может содержать несколько листов.

Сохраним несколько таблиц в один файл:

- первые строки продаж;
- справочник товаров;
- справочник регионов.

In [ ]:
multi_excel_output_path = output_dir / "loaded_data_examples.xlsx"

with pd.ExcelWriter(multi_excel_output_path, engine="openpyxl") as writer:
    sales.head(10).to_excel(writer, sheet_name="sales_sample", index=False)
    products.head(10).to_excel(writer, sheet_name="products_sample", index=False)
    regions.head(10).to_excel(writer, sheet_name="regions_sample", index=False)

print("Файл с несколькими листами сохранен:")
print(multi_excel_output_path)

## 24. Таблица-шпаргалка по функциям загрузки

| Формат | Функция pandas | Пример |
|---|---|---|
| CSV | `pd.read_csv()` | `pd.read_csv("sales.csv")` |
| Excel | `pd.read_excel()` | `pd.read_excel("products.xlsx", sheet_name="products")` |
| JSON | `pd.read_json()` | `pd.read_json("regions.json")` |
| HTML | `pd.read_html()` | `pd.read_html("web_table_sample.html")` |
| CSV-сохранение | `df.to_csv()` | `df.to_csv("result.csv", index=False)` |
| Excel-сохранение | `df.to_excel()` | `df.to_excel("result.xlsx", index=False)` |

## 25. Таблица-шпаргалка по параметрам

| Параметр | Где используется | Что делает |
|---|---|---|
| `sep` | `read_csv()` | Указывает разделитель в CSV |
| `encoding` | `read_csv()` | Указывает кодировку файла |
| `sheet_name` | `read_excel()` | Выбирает лист Excel |
| `header` | `read_csv()`, `read_excel()` | Указывает строку с заголовками |
| `index_col` | `read_csv()`, `read_excel()` | Делает указанный столбец индексом |
| `index=False` | `to_csv()`, `to_excel()` | Не сохраняет индекс в файл |

## 26. Типовые ошибки при загрузке данных

### `FileNotFoundError`

Файл не найден.

Проверьте:

```python
Path.cwd()
Path("data/raw/sales.csv").exists()
```

---

### `UnicodeDecodeError`

Проблема с кодировкой.

Попробуйте:

```python
pd.read_csv("file.csv", encoding="utf-8")
pd.read_csv("file.csv", encoding="utf-8-sig")
pd.read_csv("file.csv", encoding="cp1251")
```

---

### Один столбец вместо многих

Скорее всего, неверный разделитель.

Попробуйте:

```python
pd.read_csv("file.csv", sep=";")
pd.read_csv("file.csv", sep=",")
```

---

### `ImportError` при чтении Excel

Не установлена библиотека для Excel.

Установите зависимости:

```bash
pip install -r requirements.txt
```

---

### `ValueError: Worksheet named ... not found`

Указан неверный лист Excel.

Проверьте список листов:

```python
pd.ExcelFile("file.xlsx").sheet_names
```

## 27. Мини-задание

Выполните задания самостоятельно.

### Задание 1

Загрузите файл `clients.csv` в переменную `clients_task`.

Выведите первые 5 строк.

In [ ]:
# Ваш код здесь

### Задание 2

Загрузите лист `products` из файла `products.xlsx` в переменную `products_task`.

Выведите размер таблицы.

In [ ]:
# Ваш код здесь

### Задание 3

Загрузите файл `regions.json` в переменную `regions_task`.

Выведите список столбцов.

In [ ]:
# Ваш код здесь

### Задание 4

Загрузите HTML-таблицу из `web_table_sample.html`.

Сохраните первую таблицу в переменную `plans_task`.

Выведите первые строки.

In [ ]:
# Ваш код здесь

### Задание 5

Сохраните первые 10 строк `sales` в файл:

```text
data/output/my_sales_sample.csv
```

In [ ]:
# Ваш код здесь

### Задание 6

Сохраните первые 10 строк `products` в файл:

```text
data/output/my_products_sample.xlsx
```

In [ ]:
# Ваш код здесь

## 28. Контрольные вопросы

Ответьте своими словами:

1. Что делает `pd.read_csv()`?
2. Для чего нужен параметр `sep`?
3. Для чего нужен параметр `encoding`?
4. Что делает `sheet_name`?
5. Почему `pd.read_html()` возвращает список таблиц?
6. Что означает `header=None`?
7. Что делает `index_col`?
8. Почему при сохранении часто указывают `index=False`?
9. Чем CSV отличается от Excel?
10. Почему после загрузки данных нужно проверять `shape`, `head()` и `info()`?

## 29. Итог ноутбука

В этом ноутбуке мы научились:

- загружать CSV;
- указывать разделитель через `sep`;
- указывать кодировку через `encoding`;
- загружать Excel;
- выбирать лист Excel через `sheet_name`;
- загружать JSON;
- загружать HTML-таблицу;
- использовать `header` и `index_col`;
- сохранять данные в CSV и Excel.

Следующий шаг:

```text
02_dataframe_types_and_quality.ipynb
```

Там мы будем разбираться, как проверять типы данных, пропуски, дубликаты и ошибки после загрузки файлов.